# GPT-SoVITS Improve — Kaggle Grokking 연구

기존 Fast Minimal 노트북의 연구 브랜치 버전입니다. Dataset `pymood1/w-voice-file`, GPU T4 ×2, Internet ON 설정 후 Run All. 노트북에는 Git 동기화와 Python 진입점만 남겨 둡니다. 설치, 전처리, 학습 설정, 결과 압축은 Git 코드에서 관리합니다.


## 0. Dataset 연결만 하면 됩니다

현재 기본 우선순위는 `pymood1/w-voice-file`입니다. 저장소가 다음 경로들을 자동 탐색합니다.

```text
/kaggle/input/datasets/pymood1/w-voice-file/datas/
/kaggle/input/w-voice-file/datas/
```

둘 다 없으면 `/kaggle/input` 아래에서 음성이 들어 있는 `datas/`를 자동 탐색합니다.
`training.list`는 선택 사항입니다. **이 노트북에서 경로 코드를 수정할 필요가 없습니다.**


### 데이터 준비는 Git 코드가 담당

이전 버전의 긴 Dataset 탐색/압축해제/`training.list` 변환 셀은 제거했습니다.
`tools/kaggle_dataset.py`가 Dataset을 준비하고, `tools/kaggle_launch_webui.py`가 Preflight와 WebUI 실행까지 연결합니다.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/pymood1/GPT-SoVITS-improve.git"
REPO_DIR = Path("/kaggle/tmp/GPT-SoVITS")
BRANCH = "research/grokking-gpt-sovits-20261121"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, str(REPO_DIR)], check=True)
else:
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"Not a git repository: {REPO_DIR}")
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"],
                                     cwd=REPO_DIR, text=True).strip()
    if origin.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError(f"Unexpected repository: {origin}")
    subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH],
                   cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO_DIR, check=True)

subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO_DIR, check=True)


## 2. Preflight 및 대조 실험

Git bootstrap이 설치 및 Dataset 탐색과 Preflight를 실행한 뒤 headless 실험을 시작합니다. WebUI 공개 링크를 만들지 않습니다.


전처리 결과를 두 모델이 공유하고 서로 다른 새 학습 디렉터리를 사용합니다.


## 3. 첫 실험: GPT/S1 Grokfast OFF vs ON

두 T4에 각각 독립적인 FP32 S1 학습을 배치합니다. BERT ON, DPO OFF, S2 V5dev 고정. 세부 설정과 시간 예산은 Git 코드에서 관리합니다. 결과는 `/kaggle/working/grokking_*.zip`으로 저장합니다. 실패하거나 시간 제한에 도달해도 로그를 보존합니다.


## 실행

Git bootstrap을 실행합니다. 실패한 경우 `/kaggle/working/grokking_bootstrap_*.zip`, 학습 결과는 `/kaggle/working/grokking_*.zip`에 저장됩니다.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/tmp/GPT-SoVITS")
subprocess.run(
    [sys.executable, "-u", str(REPO_DIR / "tools/kaggle_grokking_bootstrap.py")],
    cwd=REPO_DIR, check=True,
)
